# 📊 Exploration des Données Crypto - Phase 1

Ce notebook explore les données historiques collectées depuis Binance.

## Objectifs
1. Charger et visualiser les données collectées
2. Analyser les statistiques descriptives
3. Détecter les valeurs manquantes et anomalies
4. Visualiser les tendances de prix et volumes
5. Analyser les corrélations entre cryptos
6. Calculer les rendements et volatilité

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Configuration pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Imports réussis!")
print(f"📅 Date d'exécution: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Imports réussis!
📅 Date d'exécution: 2025-11-25 16:53:17


## 1. 📁 Chargement des Données

In [2]:
# Chemin des données
data_path = Path('../data/raw/historical')

# Lister tous les fichiers CSV
csv_files = list(data_path.glob('*.csv'))

print(f"📂 Fichiers CSV trouvés: {len(csv_files)}")
print("\n📋 Liste des fichiers CSV:")
for f in csv_files:
    size_kb = f.stat().st_size / 1024
    print(f"  • {f.name} ({size_kb:.2f} KB)")

📂 Fichiers CSV trouvés: 15

📋 Liste des fichiers CSV:
  • UNI_USDT_1h_20251123.csv (43.86 KB)
  • AVAX_USDT_1h_20251123.csv (44.36 KB)
  • ETH_USDT_4h_20251121.csv (37.72 KB)
  • BTC_USDT_1h_20251121.csv (54.52 KB)
  • DOT_USDT_1h_20251123.csv (43.93 KB)
  • BTC_USDT_1h_20251123.csv (54.33 KB)
  • LINK_USDT_1h_20251123.csv (44.51 KB)
  • BTC_USDT_4h_20251121.csv (41.49 KB)
  • ADA_USDT_1h_20251123.csv (46.93 KB)
  • USDT_TRY_1h_20251121.csv (43.86 KB)
  • USDT_TRY_1h_20251123.csv (10.28 KB)
  • ETH_USDT_1h_20251123.csv (50.00 KB)
  • BNB_USDT_1h_20251123.csv (47.24 KB)
  • USDT_ARS_1h_20251121.csv (45.90 KB)
  • SOL_USDT_1h_20251123.csv (47.24 KB)


In [3]:
# Fonction pour charger tous les fichiers
def load_all_crypto_data(data_path):
    crypto_data = {}
    for csv_file in data_path.glob('*.csv'):
        try:
            df = pd.read_csv(csv_file)
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            symbol = df['symbol'].iloc[0] if 'symbol' in df.columns else csv_file.stem
            crypto_data[symbol] = df
            print(f"✓ {symbol}: {len(df)} lignes chargées")
        except Exception as e:
            print(f"✗ Erreur avec {csv_file.name}: {e}")
    return crypto_data

crypto_data = load_all_crypto_data(data_path)
print(f"\n✅ {len(crypto_data)} cryptos chargées avec succès!")

✓ UNI/USDT: 720 lignes chargées
✓ AVAX/USDT: 720 lignes chargées
✓ ETH/USDT: 540 lignes chargées
✓ BTC/USDT: 720 lignes chargées
✓ DOT/USDT: 720 lignes chargées
✓ BTC/USDT: 720 lignes chargées
✓ LINK/USDT: 720 lignes chargées
✓ BTC/USDT: 540 lignes chargées
✓ ADA/USDT: 720 lignes chargées
✓ USDT/TRY: 720 lignes chargées
✓ USDT/TRY: 168 lignes chargées
✓ ETH/USDT: 720 lignes chargées
✓ BNB/USDT: 720 lignes chargées
✓ USDT/ARS: 720 lignes chargées
✓ SOL/USDT: 720 lignes chargées

✅ 11 cryptos chargées avec succès!


## 2. 🔍 Aperçu des Données

In [5]:
# Afficher un aperçu
for symbol, df in crypto_data.items():
    print(f"\n{'='*70}")
    print(f"📊 {symbol}")
    print(f"{'='*70}")
    print(f"Période: {df['timestamp'].min()} à {df['timestamp'].max()}")
    print(f"Bougies: {len(df)}")
    display(df.head())
    display(df.describe())


📊 UNI/USDT
Période: 2025-10-24 06:00:00 à 2025-11-23 05:00:00
Bougies: 720


,timestamp,open,high,low,close,volume,symbol
0,2025-10-24 06:00:00,6.40,6.42,6.39,6.40,81030.64,UNI/USDT
1,2025-10-24 07:00:00,6.40,6.40,6.34,6.36,72516.75,UNI/USDT
2,2025-10-24 08:00:00,6.36,6.42,6.33,6.38,86691.32,UNI/USDT
3,2025-10-24 09:00:00,6.38,6.39,6.37,6.38,42895.87,UNI/USDT
4,2025-10-24 10:00:00,6.38,6.38,6.35,6.35,35200.22,UNI/USDT


,timestamp,open,high,low,close,volume
count,720,720.00,720.00,720.00,720.00,720.00
mean,2025-11-08 05:29:59.999999744,6.55,6.62,6.49,6.55,460362.78
min,2025-10-24 06:00:00,4.92,4.97,4.74,4.92,18913.11
25%,2025-10-31 17:45:00,5.86,5.89,5.83,5.86,129657.91
50%,2025-11-08 05:30:00,6.38,6.44,6.33,6.38,250230.85
75%,2025-11-15 17:15:00,7.27,7.34,7.20,7.27,485288.97
max,2025-11-23 05:00:00,9.61,10.30,9.34,9.62,7951038.14
std,NaN,0.94,0.98,0.90,0.94,793440.96



📊 AVAX/USDT
Période: 2025-10-24 06:00:00 à 2025-11-23 05:00:00
Bougies: 720


,timestamp,open,high,low,close,volume,symbol
0,2025-10-24 06:00:00,19.61,19.63,19.55,19.61,67549.53,AVAX/USDT
1,2025-10-24 07:00:00,19.61,19.63,19.40,19.49,66106.62,AVAX/USDT
2,2025-10-24 08:00:00,19.50,19.67,19.41,19.59,86671.72,AVAX/USDT
3,2025-10-24 09:00:00,19.60,19.62,19.54,19.56,46832.02,AVAX/USDT
4,2025-10-24 10:00:00,19.57,19.58,19.49,19.50,35172.25,AVAX/USDT


...

,timestamp,open,high,low,close,volume,symbol
0,2025-10-22 02:00:00,1586.90,1592.10,1584.20,1590.00,64750.00,USDT/ARS
1,2025-10-22 03:00:00,1589.90,1591.90,1587.20,1589.10,43314.00,USDT/ARS
2,2025-10-22 04:00:00,1589.40,1593.90,1588.60,1589.40,16902.00,USDT/ARS
3,2025-10-22 05:00:00,1589.40,1590.10,1588.20,1589.70,11727.00,USDT/ARS
4,2025-10-22 06:00:00,1589.70,1590.20,1589.40,1590.00,5982.00,USDT/ARS


,timestamp,open,high,low,close,volume
count,720,720.00,720.00,720.00,720.00,720.00
mean,2025-11-06 01:29:59.999999744,1501.43,1503.82,1498.61,1501.30,89877.18
min,2025-10-22 02:00:00,1404.00,1430.20,1390.00,1401.00,3307.00
25%,2025-10-29 13:45:00,1479.20,1480.85,1476.40,1479.10,29778.75
50%,2025-11-06 01:30:00,1492.05,1493.45,1490.50,1492.20,58675.00
75%,2025-11-13 13:15:00,1500.12,1501.53,1497.33,1500.03,111999.00
max,2025-11-21 01:00:00,1605.00,1607.30,1599.00,1604.00,838618.00
std,NaN,36.27,36.15,36.07,36.12,99960.12



📊 SOL/USDT
Période: 2025-10-24 08:00:00 à 2025-11-23 07:00:00
Bougies: 720


,timestamp,open,high,low,close,volume,symbol
0,2025-10-24 08:00:00,192.15,193.45,191.70,192.97,81875.79,SOL/USDT
1,2025-10-24 09:00:00,192.96,193.39,191.88,192.11,78615.47,SOL/USDT
2,2025-10-24 10:00:00,192.11,192.16,191.44,191.78,84266.67,SOL/USDT
3,2025-10-24 11:00:00,191.78,192.20,190.58,192.13,76522.11,SOL/USDT
4,2025-10-24 12:00:00,192.13,197.00,192.13,193.66,471792.92,SOL/USDT


,timestamp,open,high,low,close,volume
count,720,720.00,720.00,720.00,720.00,720.00
mean,2025-11-08 07:30:00,163.49,164.46,162.29,163.41,178332.75
min,2025-10-24 08:00:00,124.32,126.37,121.66,124.32,16164.69
25%,2025-10-31 19:45:00,141.87,142.76,140.83,141.78,89432.11
50%,2025-11-08 07:30:00,159.45,160.91,158.54,159.44,134597.43
75%,2025-11-15 19:15:00,186.41,187.03,185.43,186.36,209359.75
max,2025-11-23 07:00:00,204.76,205.33,203.91,204.75,1906040.98
std,NaN,23.30,23.24,23.37,23.32,156961.64


## 3. 📊 Visualisation des Prix

In [6]:
# Graphique des prix
fig = go.Figure()
for symbol, df in crypto_data.items():
    fig.add_trace(go.Scatter(
        x=df['timestamp'],
        y=df['close'],
        mode='lines',
        name=symbol
    ))

fig.update_layout(
    title='📈 Évolution des Prix',
    xaxis_title='Date',
    yaxis_title='Prix (USD)',
    height=600,
    template='plotly_dark'
)
fig.show()

## 🎯 Prochaines Étapes

1. Feature Engineering - Indicateurs techniques
2. Base de données - TimescaleDB
3. Modélisation ML
4. Prédictions